In [8]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("arwabasal/brain-tumor-mri-detection")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\dsapu\.cache\kagglehub\datasets\arwabasal\brain-tumor-mri-detection\versions\1


In [13]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import DataLoader
import torch.optim as optim
from torchvision import datasets, transforms
import albumentations as A

import pytorch_lightning as pl
from pytorch_lightning import LightningModule
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger

from torchmetrics.classification import MulticlassAccuracy, F1Score, ConfusionMatrix, ROC

import pytorch_optimizer as optim1
import optuna

pl.seed_everything(42)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

Seed set to 42


In [10]:
def conv_block(in_feature, out_feature, padding=1, stride=1,
             activation="relu", pool =True, maxpool=True, kernel_size=3,
             kernel_size_pool=2, pool_stride=2, batchnorm =False)-> list[nn.Sequential]:
    layers = [nn.Conv2d(in_feature, out_feature, kernel_size=kernel_size, padding=padding, stride=stride)]
    if batchnorm:
        layers.append(nn.BatchNorm2d(out_feature))
    if activation == "relu":
        layers.append(nn.ReLU())
    elif activation == "leakyrelu":
        layers.append(nn.LeakyReLU())
    elif activation == "sigmoid":
        layers.append(nn.Sigmoid())
    elif activation == 'mish':
        layers.append(nn.Mish())
    elif activation == "tanh":
        layers.append(nn.Tanh())
    if pool:
        if maxpool:
            layers.append(nn.MaxPool2d(kernel_size=kernel_size_pool, stride=pool_stride))
        else:
            layers.append(nn.AvgPool2d(kernel_size=kernel_size_pool, stride=pool_stride))
    else:
        layers.append(nn.Identity())
    return nn.Sequential(*layers)


def linear_block(in_features, out_features, activation=None, dropout=0.0, batch_norm=False):
    layers = [nn.Linear(in_features, out_features)]
    if batch_norm:
        layers.append(BatchNorm1d(out_features))
    if activation == 'relu':
        layers.append(nn.ReLU())
    elif activation == 'sigmoid':
        layers.append(nn.Sigmoid())
    elif activation == 'mish':
        layers.append(nn.Mish())
    elif activation == 'tanh':
        layers.append(nn.Tanh())
    elif activation == 'leakyrelu':
        layers.append(nn.LeakyReLU())
    elif activation == 'softmax':
        layers.append(nn.Softmax(dim=1))
    elif activation == 'elu':
        layers.append(nn.ELU())
    elif activation == 'selu':
        layers.append(nn.SELU())
    elif activation == 'lsoftmax':
        layers.append(nn.LogSoftmax(dim=1))
    if dropout > 0.0:
        layers.append(nn.Dropout(dropout))
    return nn.Sequential(*layers)

In [11]:
class CNN(nn.Module):
    def __init__(self, input_size=0, num_classes) -> None:
        super().__init__()

        self.conv = nn.Sequential(
            conv_block(3, 8),
            conv_block(8, 16),
            conv_block(16, 32),
            nn.Flatten()
        )
        self.fc = nn.Sequential(
            linear_block(32 * 14* 14, 2**14),
            linear_block(2**14, 2**12, activation='mish'),
            linear_block(2**12, 2**10, activation='mish'),
            linear_block(2**10, 2**9, activation='mish'),
            linear_block(2**9, 2**8, activation='mish'),
            linear_block(256, num_classes, activation=None)
        )

    def forward(self, X):
        return self.fc(self.conv(X))

SyntaxError: non-default argument follows default argument (1907490478.py, line 2)

In [ ]:
class ANN(pl.LightningModule):
    def __init__(self, input_size, learning_rate, pos_weight_tensor, num_classes):
        super(ANN, self).__init__()
        self.learning_rate = learning_rate
        self.save_hyperparameters()
        self.fc = CNN(input_size=input_size, num_classes=num_classes)
        self.criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
        self.f1 = F1Score(task='binary')

    def forward(self, x):
        return self.fc(x)

    def _common_step(self, batch, batch_idx):
        images, labels = batch
        logits = self(images)
        loss = self.criterion(logits, labels)
        return loss, logits, labels

    def training_step(self, batch, batch_idx):
        inputs, labels = batch
        labels = labels.unsqueeze(1)
        outputs = self(inputs)
        loss = self.criterion(outputs, labels)
        f1 = self.f1(outputs, labels)
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
        self.log('train_f1', f1, on_step=True, on_epoch=True, prog_bar=True, logger=True)
        return loss

    def validation_step(self, batch, batch_idx):
        inputs, labels = batch
        labels = labels.unsqueeze(1)
        outputs = self(inputs)
        loss = self.criterion(outputs, labels)
        f1 = self.f1(outputs, labels)
        self.log('val_loss', loss, on_epoch=True, prog_bar=True, logger=True)
        self.log('val_f1', f1, on_epoch=True, prog_bar=True, logger=True)

    def test_step(self, batch, batch_idx):
        inputs, labels = batch
        labels = labels.unsqueeze(1)
        outputs = self(inputs)
        loss = self.criterion(outputs, labels)
        f1 = self.f1(outputs, labels)
        self.log('test_loss', loss, on_epoch=True, prog_bar=True, logger=True)
        self.log('test_f1', f1, on_epoch=True, prog_bar=True, logger=True)

    def backward(self, loss, *args, **kwargs):
        loss.backward(create_graph=True)

    def configure_optimizers(self):
        optimizer = optim1.AdaHessian(self.parameters(), lr=self.learning_rate, betas=(0.9, 0.999), weight_decay=1e-4)
        return optimizer

    def predict_step(self, batch, batch_idx, dataloader_idx=None):
        if isinstance(batch, list) or isinstance(batch, tuple):
            inputs, _ = batch
        else:
            inputs = batch
        return self(inputs)

In [17]:
IMG_SIZE = 224
bs = 32
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=20, p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.7),
    A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.5),
    A.CoarseDropout(max_holes=8, max_height=IMG_SIZE//8, max_width=IMG_SIZE//8, 
                    min_holes=1, fill_value=0, p=0.5),
    A.Normalize(mean=MEAN, std=STD),
    A.ToTensorV2()
])

val_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=MEAN, std=STD),
    A.ToTensorV2()
])